<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/01_basics/01_minimal_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. init / log / finish の基本

## この章のゴール
実験条件を `config` に残し、metric を `run.log()` で記録します。

## 注目する W&B コード
```python
with wandb.init(project=PROJECT, config=cfg) as run:
    run.log({"train/loss": loss, "epoch": epoch})
```

In [ ]:
# リポジトリ内ではルートを使い、Colab 単体では共通コードを取得します。
from pathlib import Path
import os
import sys
import urllib.request

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").is_file()), Path.cwd())
if not (root / "course_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py",
        root / "course_utils.py",
    )
os.chdir(root)
sys.path.insert(0, str(root))

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb()

## 1 run を記録する
まずは、実験 1 回分を 1 run として記録します。ここでは学習そのものは簡単な数値列にして、W&B の書き方に集中します。

In [ ]:
import wandb

cfg = {"lr": 0.1, "epochs": 5, "model": "tiny-demo"}
losses = [0.90, 0.72, 0.58, 0.49, 0.43]

with wandb.init(**WANDB_INIT, name="01-basic-run", config=cfg, tags=["course", "basic"]) as run:
    for epoch, loss in enumerate(losses):
        run.log({"train/loss": loss, "epoch": epoch})
    run_url = run.url

print("Run URL:", run_url)

## 比較用に2 runsを追加する
学習率を変えた2 runsを作ります。この章全体では合計3 runsになります。数値はログの練習用で、モデルの学習結果ではありません。


In [ ]:
configs = [
    {"lr": 0.1, "epochs": 5, "model": "tiny-demo"},
    {"lr": 0.03, "epochs": 5, "model": "tiny-demo"},
]

for cfg in configs:
    with wandb.init(**WANDB_INIT, name=f"01-compare-lr-{cfg['lr']}", config=cfg, tags=["course", "compare"]) as run:
        for epoch in range(cfg["epochs"]):
            loss = 1.0 / (epoch + 1) + cfg["lr"]
            run.log({"train/loss": loss, "epoch": epoch})

print("2 runs logged")

## W&B App で確認すること
- Runs table に run が並ぶ
- `config.lr` が比較軸になる
- `train/loss` の chart で run ごとの違いを見られる